# Peut-on prédire le succès d'une startup dès sa première phrase de pitch ?

*Capucine, Gaspard, Romain — Python pour la data science*

Ce notebook s'exécute de la première à la dernière cellule sans erreur, en repartant des copies
datées des données brutes stockées sur S3. Les fonctions sont dans `src/`.

## Paramètres

`COLLECTER = False` : le notebook lit la copie des données du 30/10/2026 sur le stockage S3 du SSP Cloud, celle que commente le rapport. Avec `COLLECTER = True`, il retélécharge les données depuis les API : les chiffres peuvent alors différer un peu du texte.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from src import analyse, collecte, graphiques, nettoyage, succes, texte
from src.config import DATE_COPIE

COLLECTER = False  # True : données à jour depuis les API au lieu de la copie du DATE_COPIE
GRAINE = 2026  # rend reproductibles les tirages aléatoires (intervalles de confiance par bootstrap)
rng = np.random.default_rng(GRAINE)

# Graphiques Plotly visibles dans VSCode ou Jupyter comme dans la version en ligne (nbviewer).
pio.renderers.default = "plotly_mimetype+notebook_connected"

texte.preparer_nltk()
print(f"Copie des données : {DATE_COPIE} ; collecte depuis les API : {COLLECTER}")

## Données

La base Y Combinator et les posts Launch HN sont appariés pour former la table commune : une ligne par startup qui a publié un Launch HN, avec son pitch d'origine et son statut actuel.

In [ ]:
yc = collecte.charger_brut("yc", collecter=COLLECTER)
hn = collecte.charger_brut("hn", collecter=COLLECTER)
table = nettoyage.joindre(yc, hn)

print(f"Base YC : {len(yc)} startups ; posts Launch HN : {len(hn)} ; "
      f"table commune : {len(table)} startups appariées")
table.head()

## Introduction

- Accroche : un analyste VC lit des centaines de pitchs ; Y Combinator demande aux fondateurs de résumer leur projet en une phrase.
- La question et les deux hypothèses opposées.
- Annonce du plan.

## 1. Construire les données

Aucune base ne relie directement le pitch d'une startup à son destin : on la construit en appariant
deux sources publiques, puis on restreint l'étude aux startups dont on connaît vraiment l'issue.

### 1.1 Deux sources complémentaires

| Source | Ce qu'on en tire | Accès |
|---|---|---|
| Base Y Combinator (API yc-oss) | Une ligne par startup : nom, anciens noms, promotion, statut actuel (*Active*, *Inactive*, *Acquired*, *Public*), secteur, région, taille de l'équipe, stade, pitch actuel (`one_liner`) | Fichier JSON public, sans clé |
| Posts « Launch HN » (API Algolia de Hacker News) | Une ligne par post : titre « Launch HN: Nom (YC W20) – pitch », date, points, commentaires | API de recherche publique, 1 000 résultats par requête, pagination par date |

Y Combinator donne le destin de chaque startup, Hacker News donne la phrase avec laquelle elle s'est
présentée au public. Les deux copies analysées datent du 30/10/2026 (`DATE_COPIE`).

In [ ]:
promotions_yc = yc["batch"].map(nettoyage.debut_promotion)
dates_hn = pd.to_datetime(hn["created_at"])
pd.DataFrame(
    {
        "lignes": [len(yc), len(hn)],
        "unité": ["startup", "post"],
        "de": [promotions_yc.min().year, dates_hn.min().year],
        "à": [promotions_yc.max().year, dates_hn.max().year],
    },
    index=["Base Y Combinator (promotions)", "Posts Launch HN (dates)"],
)

La base YC couvre toutes les promotions depuis 2005, alors que les Launch HN n'existent en nombre
qu'à partir de la fin des années 2010 : c'est Hacker News qui limite la taille de l'échantillon.
Seule une partie des startups YC publie un Launch HN, ce qui pose la question de la
représentativité (section 1.5).

### 1.2 Pourquoi le pitch d'origine

La base YC contient bien une phrase de présentation, le `one_liner`, mais c'est celle d'aujourd'hui :
une startup la réécrit à mesure qu'elle pivote ou grandit. Mettre en regard ce pitch actuel et le
succès serait circulaire, puisqu'une startup qui a réussi décrit ce qu'elle est devenue. Le titre du
post Launch HN, publié en général quelques mois après le début de la promotion, fige au contraire la
phrase des débuts.

In [ ]:
exemples = ["Substack", "PostHog", "Hightouch", "Vapi"]
display(
    table.loc[
        table["nom"].isin(exemples), ["nom", "promotion", "statut", "pitch_origine", "pitch_actuel"]
    ]
)

def _simplifier(pitchs: pd.Series) -> pd.Series:
    return pitchs.str.lower().str.strip(" .")

identiques = _simplifier(table["pitch_origine"]) == _simplifier(table["pitch_actuel"])
print(f"Pitch actuel identique au pitch d'origine : {identiques.sum()} startups sur {len(table)} "
      f"({identiques.mean():.0%})")

Seuls 3 % des pitchs n'ont pas changé, à la casse et à la ponctuation près. Les réécritures vont de la
reformulation (Substack : « Paid email newsletters made simple » devient « A place for independent
writing ») au pivot complet (Carry, qui réservait des voyages sur Slack, est devenue Hightouch, une
plateforme de données clients). C'est donc bien le pitch d'origine qu'il faut étudier.

### 1.3 Collecte et jointure

`collecte.telecharger_launch_hn` récupère tous les posts du tag `launch_hn`, par paquets de 1 000 en
remontant le temps. `nettoyage.extraire_titre` découpe chaque titre en nom, promotion (« W20 » devient
« Winter 2020 ») et pitch. `nettoyage.joindre` apparie ensuite chaque post à la base YC sur le nom
normalisé (minuscules, sans accents ni suffixes comme « Inc » ou « AI ») et la promotion, puis, pour
les startups renommées, sur leurs anciens noms. Une startup lancée deux fois garde son post le plus
ancien.

Les posts dont le titre n'indique pas de promotion sont écartés : sans elle, deux startups homonymes
de promotions différentes seraient confondues, et le risque d'apparier un post à la mauvaise
entreprise est plus grave que la perte de ces posts.

In [ ]:
bilan = nettoyage.bilan_jointure(table, hn)
sans_promotion = int(bilan["posts"].get("inconnue", 0))
non_reconnus = int(hn["title"].map(lambda titre: nettoyage.extraire_titre(titre)["nom"] is None).sum())
avec_promotion = len(hn) - sans_promotion
print(f"{len(hn)} posts Launch HN, dont {sans_promotion} sans promotion dans le titre "
      f"({non_reconnus} au format non reconnu) : écartés")
print(f"{len(table)} startups appariées : {len(table) / len(hn):.0%} des posts, "
      f"{len(table) / avec_promotion:.0%} des posts avec promotion")

par_annee = (
    bilan.drop(index="inconnue", errors="ignore")
    .groupby(lambda promotion: int(promotion.split()[1]))[["posts", "appariements"]]
    .sum()
    .rename_axis("année de promotion")
)
par_annee["taux"] = (par_annee["appariements"] / par_annee["posts"]).round(2)
par_annee.T

Trois posts sur quatre aboutissent à une startup (74 %). L'essentiel des pertes vient des 222 posts
sans promotion lisible ; parmi les posts qui en ont une, 91 % sont appariés, et une partie du reste
correspond à des startups lancées deux fois, qui ne comptent qu'une fois.

### 1.4 Période d'étude et échantillon final

Le statut *Inactive* n'a de sens qu'avec du recul : une startup de la promotion 2025 n'a pas encore
eu le temps d'échouer. Le graphique compare la part d'inactives par année de promotion dans toute la
base YC et dans la table commune.

In [ ]:
graphiques.courbe_recul(yc, table)

Dans la base YC, la part d'inactives tourne autour de 30 % jusqu'en 2016, puis baisse presque chaque
année jusqu'à 1 ou 2 % pour les promotions 2025 et 2026, et la table commune suit la même pente : sur
les promotions récentes, comparer des taux d'échec reviendrait à comparer des âges. On garde les
promotions 2017 à 2021, qui ont au moins cinq ans de recul ; avant 2017, les Launch HN sont trop rares
(8 posts). Même sur cette fenêtre, la part d'inactives baisse encore (28 % pour 2017 contre 18 % pour
2021 dans la base YC) : c'est pourquoi les comparaisons suivantes se font aussi à promotion égale.

In [ ]:
etude, etapes = nettoyage.echantillon_etude(table, debut=2017, fin=2021, delai_max=1.0)
etapes.round(3)

In [ ]:
annees = table["promotion"].map(nettoyage.debut_promotion).dt.year
periode = table[annees.between(2017, 2021)]
moment = (
    (periode["hn_delai_ans"] > 1)
    .map({False: "Post dans l'année", True: "Post plus d'un an après"})
    .rename("moment du post")
)
statuts = pd.crosstab(moment, periode["statut"], normalize="index").mul(100).round(0)
statuts.insert(0, "startups", moment.value_counts())
print(f"Délai médian entre le début de la promotion et le post : "
      f"{etude['hn_delai_ans'].median() * 12:.1f} mois")
statuts

La plupart des startups publient leur Launch HN pendant ou juste après la promotion (délai médian
d'un peu plus de deux mois). On écarte les 79 posts publiés plus d'un an après, pour deux raisons :
leur pitch n'est plus celui des débuts, et seules les startups encore en vie peuvent se lancer
tardivement. Ces posts tardifs comptent d'ailleurs moins d'inactives (18 % contre 23 %) : les garder
gonflerait artificiellement la survie. L'échantillon d'étude compte 392 startups, dont 90 échecs
(23 %).

### 1.5 Représentativité

Les startups qui font un Launch HN ne sont pas tirées au hasard parmi les startups YC. On compare
l'échantillon à toutes les startups YC des promotions 2017 à 2021 (pourcentages).

In [ ]:
yc_periode = yc[promotions_yc.dt.year.between(2017, 2021)]
print(f"Startups YC 2017-2021 : {len(yc_periode)} ; échantillon : {len(etude)} "
      f"({len(etude) / len(yc_periode):.0%})")


def comparer(colonne_yc: str, colonne_etude: str) -> pd.DataFrame:
    """Répartition (en %) d'une variable dans la base YC 2017-2021 et dans l'échantillon."""
    return (
        pd.concat(
            {
                "YC 2017-2021": yc_periode[colonne_yc].value_counts(normalize=True),
                "Échantillon": etude[colonne_etude].value_counts(normalize=True),
            },
            axis=1,
        )
        .fillna(0)
        .mul(100)
        .round(1)
    )


display(comparer("status", "statut"))
display(comparer("industry", "secteur"))

L'échantillon représente environ une startup YC sur cinq de la période. Son taux d'échec est le même
que celui de la base (23 %), ce qui écarte un biais grossier sur la variable étudiée ; les rachats y
sont un peu plus fréquents (22 % contre 16 %). Le secteur est plus déformé : le B2B, dont les
fondateurs visent un public de développeurs très présent sur Hacker News, pèse 55 % de l'échantillon
contre 42 % dans YC, la santé deux fois moins (9 % contre 16 %). Les résultats valent donc pour
« les startups YC qui ont fait un Launch HN », surtout B2B, et pas pour toutes les startups.

## 2. Qui réussit ? Définir et décrire le succès

### 2.1 Définir la réussite

La variable étudiée dans tout le rapport est l'échec : une startup a échoué si la base YC la dit
*Inactive* (`succes.echec`). C'est la seule information de destin disponible pour toutes les
startups et sans ambiguïté. Pour nuancer, `succes.niveau_succes` classe aussi les startups en quatre
niveaux : échec, survie (active), croissance (active, au stade *Growth* ou avec une équipe plus
grande que la médiane des actives de sa promotion) et sortie (rachetée ou en bourse). Le seuil
d'équipe dépend de la promotion parce qu'une startup plus ancienne a eu plus de temps pour embaucher.

In [ ]:
etude["echec"] = succes.echec(etude)
etude["niveau_succes"] = succes.niveau_succes(etude)

niveaux = etude["niveau_succes"].value_counts(sort=False).to_frame("startups")
niveaux["part (%)"] = (niveaux["startups"] / len(etude) * 100).round(0)
display(niveaux)

seuils = etude["taille_equipe"].where(etude["statut"] == "Active").groupby(etude["promotion"]).median()
sauvetage = succes.rachats_de_sauvetage(etude)
print(f"Seuil d'équipe de la croissance : de {seuils.min():.0f} à {seuils.max():.0f} personnes "
      "selon la promotion")
print(f"Rachats de sauvetage (5 personnes ou moins) : {sauvetage.sum()} "
      f"sur {(etude['statut'] == 'Acquired').sum()} rachats")

Les quatre niveaux ont des tailles comparables (de 87 à 122 startups). Le seuil d'équipe va de
10 personnes (Summer 2021, la promotion la plus récente) à 80 (Winter 2018) et il est en général plus
haut pour les promotions anciennes, ce qui justifie de le calculer par promotion. Plus d'un rachat sur trois (32 sur 87) concerne une équipe de cinq personnes
ou moins : ces rachats ressemblent plus à un recrutement ou à une sortie de secours qu'à une réussite.
C'est une raison de plus pour s'en tenir à l'échec, qui ne dépend pas de cette frontière, comme
variable principale.

### 2.2 Contrôles qualité

In [ ]:
succes.controles_qualite(etude).round(3)

Le statut, dont dépend l'échec, ne manque jamais, aucune startup n'apparaît deux fois et toutes les
promotions sont reconnues. Les seuls trous concernent la taille d'équipe (3 % des startups, qui ne
peuvent alors être classées en croissance que par leur stade) et la région (1 %, regroupées en
« Non renseigné »).

### 2.3 Taux d'échec par secteur et par région

Chaque barre donne la part de startups inactives et son intervalle de confiance à 95 % (Wilson) ;
l'effectif est entre parenthèses, et les secteurs de moins de 10 startups sont regroupés en
« Autres ».

In [ ]:
taux_secteur = analyse.taux_echec(etude, "secteur")
graphiques.barres_taux(
    taux_secteur, "Les startups grand public échouent deux fois plus souvent que la moyenne"
)

In [ ]:
analyse.taux_echec_standardise(etude, "secteur").round(2)

Le secteur grand public (*Consumer*) se détache : 52 % d'échecs, contre 23 % pour l'ensemble, et son
intervalle ne recoupe pas celui du B2B (19 %). À promotion égale (tableau, standardisation
indirecte), il compte 2,2 fois plus d'échecs qu'attendu, avec un intervalle de 1,5 à 3,1 : l'écart
ne vient ni du hasard ni de l'âge des startups. À l'autre bout, la santé échoue peu (9 %, ratio de
0,4), et l'éducation ne compte aucun échec, mais sur 12 startups seulement.

In [ ]:
taux_region = analyse.taux_echec(etude, "region")
graphiques.barres_taux(taux_region, "Hors Amérique du Nord, les effectifs sont trop faibles pour conclure")

Trois startups sur quatre de l'échantillon sont nord-américaines (293 sur 392). Leur taux d'échec
(25 %) est un peu plus haut que celui des startups européennes (14 %), mais les intervalles se
recouvrent largement, et les autres régions comptent moins de 15 startups chacune : la région ne
distingue pas les startups de façon fiable, et on ne la retiendra que comme variable de contrôle.

### 2.4 La barre à dépasser

In [ ]:
taux_promotion = analyse.taux_echec(etude, "promotion")
secteurs = taux_secteur.drop(index=[analyse.AUTRES, analyse.NON_RENSEIGNE], errors="ignore")
for nom, taux in [("Secteur", secteurs), ("Promotion", taux_promotion)]:
    haut, bas = taux["taux"].idxmax(), taux["taux"].idxmin()
    print(f"{nom} : de {taux.loc[bas, 'taux']:.0%} ({bas}, {taux.loc[bas, 'effectif']} startups) "
          f"à {taux.loc[haut, 'taux']:.0%} ({haut}, {taux.loc[haut, 'effectif']} startups)")

Sans lire un seul mot du pitch, le secteur fait déjà passer le taux d'échec de 0 à 52 %, et la
promotion de 5 % (Winter 2019) à 42 % (Winter 2018). Un mot qui paraît associé à l'échec peut donc
simplement signaler un secteur risqué (« app », « consumers ») ou une promotion plus ancienne. C'est la
barre à dépasser : dans les parties 3 et 4, un trait du pitch ne compte que s'il reste associé à
l'échec à promotion et secteur égaux.

## 3. Que disent les pitchs ?

### 3.1 Anatomie d'un pitch
### 3.2 Les mots du succès et de l'échec
### 3.3 Le jargon nuit-il ?
### 3.4 Les gagnants réécrivent-ils leur pitch ?
### 3.5 Et la foule ? Réaction sur Hacker News

## 4. Le pitch prédit-il mieux que le secteur ?

### 4.1 Modèle de référence : secteur, promotion, région
### 4.2 Modèle avec le texte du pitch
### 4.3 Résultat

## Conclusion

- La réponse à la question, en une phrase chiffrée.
- Les limites : biais de survie, couverture de Launch HN, startups « actives » sans croissance, échantillon sélectionné par YC, corrélation et causalité.
- Ouverture.

## Annexes

### A. Dictionnaire des variables de la table commune
### B. Relancer le projet